In [1]:
class Ambulance:
    def __init__(self, ambulance_id: str, station_id: str):
        self.id = ambulance_id
        self.station_id = station_id
        self.state = "Available"
        self.current_location = station_id


class Incident:
    def __init__(self, incident_id: int, location: str, arrival_time: float):
        self.id = incident_id
        self.location = location
        self.arrival_time = arrival_time
        self.status = "Pending"


class RoadNetwork:
    def __init__(self):
        self.roads = {}
        self.route_allocations = {}

    def add_road(self, u: str, v: str, cost: float, is_open: bool = True, max_capacity: int = float('inf')):
        self.roads[(u, v)] = {
            "cost": cost,
            "is_open": is_open,
            "max_capacity": max_capacity
        }
        if (u, v) not in self.route_allocations:
            self.route_allocations[(u, v)] = 0

    def is_path_valid(self, u: str, v: str) -> bool:
        """Validates route availability based on closures and route capacity limits."""
        if (u, v) not in self.roads:
            return False

        road = self.roads[(u, v)]

        if not road["is_open"]:
            return False

        current_count = self.route_allocations.get((u, v), 0)
        if current_count >= road["max_capacity"]:
            return False

        return True

    def increment_allocation(self, u: str, v: str):
        if self.is_path_valid(u, v):
            self.route_allocations[(u, v)] += 1
            return True
        return False


class DispatcherAgent:
    def __init__(self, ambulances: list, road_network: RoadNetwork):
        self.ambulances = ambulances
        self.road_network = road_network

    def dispatch_ambulance(self, incident: Incident):
        """Finds and assigns the optimal available ambulance respecting operational constraints."""
        best_ambulance = None
        min_cost = float('inf')

        print(f"\n--- [Dispatch Attempt] Incident #{incident.id} at Location '{incident.location}' ---")

        for amb in self.ambulances:
            if amb.state == "Available":
                start_node = amb.station_id
                target_node = incident.location

                if self.road_network.is_path_valid(start_node, target_node):
                    road_info = self.road_network.roads.get((start_node, target_node))
                    cost = road_info["cost"] if road_info else float('inf')

                    if cost < min_cost:
                        min_cost = cost
                        best_ambulance = amb

        if best_ambulance:
            start_node = best_ambulance.station_id
            target_node = incident.location
            best_ambulance.state = "Dispatched"
            best_ambulance.current_location = target_node
            incident.status = "Assigned"
            self.road_network.increment_allocation(start_node, target_node)
            print(f"  ✔ [SUCCESS] Assigned Ambulance [{best_ambulance.id}] from Station [{best_ambulance.station_id}] (Cost = {min_cost})")
            return best_ambulance
        else:
            print(f"    [FAILED] No available ambulance with a valid route for Incident #{incident.id}!")
            return None


if __name__ == "__main__":
    network = RoadNetwork()

    cost_matrix = {
        "S1": {"A1": 24, "A2": 45, "A3": 37, "A4": 18, "A5": 19},
        "S2": {"A1": 28, "A2": 36, "A3": 37, "A4": 13, "A5": 23},
        "S3": {"A1": 45, "A2": 39, "A3": 31, "A4": 42, "A5": 15},
        "S4": {"A1": 26, "A2": 42, "A3": 19, "A4": 19, "A5": 18}
    }

    for station, destinations in cost_matrix.items():
        for dest, cost in destinations.items():
            network.add_road(station, dest, cost=cost, is_open=True, max_capacity=float('inf'))

    network.add_road("S1", "A4", cost=18, is_open=False, max_capacity=float('inf'))
    network.add_road("S3", "A5", cost=15, is_open=True, max_capacity=4)

    supply_data = {
        "S1": 8,
        "S2": 6,
        "S3": 7,
        "S4": 9
    }

    ambulances_list = []
    for station_id, count in supply_data.items():
        for i in range(1, count + 1):
            ambulances_list.append(Ambulance(ambulance_id=f"{station_id}_Amb{i}", station_id=station_id))

    dispatcher = DispatcherAgent(ambulances=ambulances_list, road_network=network)

    incidents_list = [
        Incident(incident_id=501, location="A4", arrival_time=1.0),
        Incident(incident_id=502, location="A5", arrival_time=2.0),
        Incident(incident_id=503, location="A5", arrival_time=3.0),
        Incident(incident_id=504, location="A5", arrival_time=4.0),
        Incident(incident_id=505, location="A5", arrival_time=5.0),
        Incident(incident_id=506, location="A5", arrival_time=6.0),
    ]

    print("==================================================================")
    print("    AMBULANCE ALLOCATION SIMULATION (CONSTRAINED ROUTE NETWORK)   ")
    print("==================================================================")

    for inc in incidents_list:
        dispatcher.dispatch_ambulance(inc)

    print("\n==================================================================")
    print("               SIMULATION COMPLETED SUCCESSFULLY                  ")
    print("==================================================================")

    AMBULANCE ALLOCATION SIMULATION (CONSTRAINED ROUTE NETWORK)   

--- [Dispatch Attempt] Incident #501 at Location 'A4' ---
  ✔ [SUCCESS] Assigned Ambulance [S2_Amb1] from Station [S2] (Cost = 13)

--- [Dispatch Attempt] Incident #502 at Location 'A5' ---
  ✔ [SUCCESS] Assigned Ambulance [S3_Amb1] from Station [S3] (Cost = 15)

--- [Dispatch Attempt] Incident #503 at Location 'A5' ---
  ✔ [SUCCESS] Assigned Ambulance [S3_Amb2] from Station [S3] (Cost = 15)

--- [Dispatch Attempt] Incident #504 at Location 'A5' ---
  ✔ [SUCCESS] Assigned Ambulance [S3_Amb3] from Station [S3] (Cost = 15)

--- [Dispatch Attempt] Incident #505 at Location 'A5' ---
  ✔ [SUCCESS] Assigned Ambulance [S3_Amb4] from Station [S3] (Cost = 15)

--- [Dispatch Attempt] Incident #506 at Location 'A5' ---
  ✔ [SUCCESS] Assigned Ambulance [S4_Amb1] from Station [S4] (Cost = 18)

               SIMULATION COMPLETED SUCCESSFULLY                  
